# Churn Prediction Model & Model Card

This notebook trains a leakage-safe churn model using the provided modeling snapshot.

In [1]:
!git clone https://github.com/Vivek-data-scientist/d2c-churn-part3-churn-model.git
%cd d2c-churn-part3-churn-model

Cloning into 'd2c-churn-part3-churn-model'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 26 (delta 2), reused 26 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 373.07 KiB | 8.88 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/d2c-churn-part3-churn-model


In [3]:
import json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
DATA = Path('data/raw')
SNAPSHOT = pd.Timestamp('2025-09-30')
rfm = pd.read_csv(DATA/'rfm_modeling_snapshot.csv', parse_dates=['snapshot_date'])
orders = pd.read_csv(DATA/'orders.csv', parse_dates=['order_date'])
rfm.shape, rfm.split.value_counts().to_dict(), rfm.churn_next_60d.mean()

((2400, 29),
 {'train': 1728, 'validation': 336, 'test': 336},
 np.float64(0.46958333333333335))

## Leakage Checks

The raw `orders.csv` includes post-snapshot rows, but this notebook uses the provided modeling snapshot as its feature table. Target and split columns are excluded from predictors.

In [4]:
feature_cols = [c for c in rfm.columns if c not in ['customer_id','snapshot_date','churn_next_60d','split']]
leakage_audit = {
    'post_snapshot_orders_in_raw_file': int((orders.order_date > SNAPSHOT).sum()),
    'target_excluded': 'churn_next_60d' not in feature_cols,
    'split_excluded': 'split' not in feature_cols,
    'feature_count': len(feature_cols),
}
leakage_audit

{'post_snapshot_orders_in_raw_file': 1872,
 'target_excluded': True,
 'split_excluded': True,
 'feature_count': 25}

## Train / Validation / Test Split

In [5]:
train = rfm[rfm.split == 'train'].copy()
validation = rfm[rfm.split == 'validation'].copy()
test = rfm[rfm.split == 'test'].copy()
pd.DataFrame({
    'split': ['train','validation','test'],
    'rows': [len(train), len(validation), len(test)],
    'churn_rate': [train.churn_next_60d.mean(), validation.churn_next_60d.mean(), test.churn_next_60d.mean()],
})

,split,rows,churn_rate
0,train,1728,0.469907
1,validation,336,0.437500
2,test,336,0.500000


## Baseline and Final Model Metrics

In [6]:
metrics = json.load(open('metrics.json'))
metrics

{'leakage_prevention': {'snapshot_date': '2025-09-30',
  'modeling_snapshot_rows': 2400,
  'target_column_excluded_from_features': True,
  'split_column_excluded_from_features': True,
  'post_snapshot_orders_in_raw_file': 1872,
  'feature_source': 'rfm_modeling_snapshot.csv',
  'feature_source_leakage_note': 'Data dictionary states the snapshot table contains only features available on or before 2025-09-30.',
  'feature_count': 25},
 'baseline_model': {'name': 'train_prevalence_baseline',
  'description': 'Predicts the train-set churn prevalence for every customer.',
  'train_churn_prevalence': 0.4699074074074074,
  'threshold': 0.5,
  'metrics_by_split': {'train': {'accuracy': 0.5300925925925926,
    'precision': 0.0,
    'recall': 0.0,
    'f1': 0.0,
    'roc_auc': 0.5,
    'pr_auc': 0.4699074074074074,
    'log_loss': 0.6913349573195646,
    'tp': 0,
    'fp': 0,
    'tn': 916,
    'fn': 812},
   'validation': {'accuracy': 0.5625,
    'precision': 0.0,
    'recall': 0.0,
    'f1': 0

In [7]:
summary_rows = []
for model_name in ['baseline_model', 'final_model']:
    for split, vals in metrics[model_name]['metrics_by_split'].items():
        summary_rows.append({'model': model_name, 'split': split, **{k: vals[k] for k in ['accuracy','precision','recall','f1','roc_auc','pr_auc','tp','fp','tn','fn']}})
pd.DataFrame(summary_rows)

,model,split,accuracy,precision,recall,f1,roc_auc,pr_auc,tp,fp,tn,fn
0,baseline_model,train,0.530093,0.000000,0.000000,0.000000,0.500000,0.469907,0,0,916,812
1,baseline_model,validation,0.562500,0.000000,0.000000,0.000000,0.500000,0.437500,0,0,189,147
2,baseline_model,test,0.500000,0.000000,0.000000,0.000000,0.500000,0.500000,0,0,168,168
3,final_model,train,0.752894,0.671111,0.929803,0.779556,0.886511,0.858430,755,370,546,57
4,final_model,validation,0.747024,0.647619,0.925170,0.761905,0.882374,0.867500,136,74,115,11
5,final_model,test,0.738095,0.665289,0.958333,0.785366,0.884602,0.877905,161,81,87,7


## Threshold Selection

The final threshold is chosen on validation data using a business score that weights recall and precision while penalizing outreach volume.

In [8]:
pd.read_csv('outputs/threshold_search.csv').head(10)

,threshold,business_score,outreach_rate,accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss,tp,fp,tn,fn
0,0.24,0.761701,0.625000,0.747024,0.647619,0.925170,0.761905,0.882374,0.8675,0.423821,136,74,115,11
1,0.32,0.761379,0.538690,0.785714,0.707182,0.870748,0.780488,0.882374,0.8675,0.423821,128,53,136,19
2,0.39,0.760674,0.491071,0.803571,0.745455,0.836735,0.788462,0.882374,0.8675,0.423821,123,42,147,24
3,0.31,0.759544,0.541667,0.782738,0.703297,0.870748,0.778116,0.882374,0.8675,0.423821,128,54,135,19
4,0.38,0.756571,0.497024,0.797619,0.736527,0.836735,0.783439,0.882374,0.8675,0.423821,123,44,145,24
5,0.40,0.756237,0.488095,0.800595,0.743902,0.829932,0.784566,0.882374,0.8675,0.423821,122,42,147,25
6,0.37,0.754549,0.500000,0.794643,0.732143,0.836735,0.780952,0.882374,0.8675,0.423821,123,45,144,24
7,0.22,0.754255,0.651786,0.726190,0.625571,0.931973,0.748634,0.882374,0.8675,0.423821,137,82,107,10
8,0.30,0.754135,0.550595,0.773810,0.691892,0.870748,0.771084,0.882374,0.8675,0.423821,128,57,132,19
9,0.28,0.753328,0.562500,0.767857,0.682540,0.877551,0.767857,0.882374,0.8675,0.423821,129,60,129,18


## Feature Importance

In [9]:
pd.read_csv('outputs/feature_importance.csv').head(20)

,feature,coefficient,abs_coefficient,direction
0,recency_days,1.756800,1.756800,increases churn risk
1,monetary_180d,-0.436657,0.436657,decreases churn risk
2,preferred_category=Fragrance,-0.394286,0.394286,decreases churn risk
3,acquisition_channel=Organic,-0.391847,0.391847,decreases churn risk
4,return_rate_180d,0.347311,0.347311,increases churn risk
5,ticket_count_90d,-0.313807,0.313807,decreases churn risk
6,loyalty_tier=Platinum,-0.309626,0.309626,decreases churn risk
7,negative_ticket_rate_90d,0.306327,0.306327,increases churn risk
8,avg_discount_pct_180d,0.298314,0.298314,increases churn risk
9,last_visit_days_ago,0.277984,0.277984,increases churn risk


## Error Analysis Examples

In [10]:
pd.read_csv('outputs/error_examples.csv')

,error_type,customer_id,churn_probability,predicted_churn,churn_next_60d,recency_days,frequency_180d,monetary_180d,return_rate_180d,avg_discount_pct_180d,ticket_count_90d,negative_ticket_rate_90d,sessions_30d,campaign_clicks_30d,last_visit_days_ago,loyalty_tier,marketing_consent
0,False Positive,CUST01246,0.982330,1,0,262,0,0.00,0.0,0.000,0,0.0,1,3,60,Silver,Yes
1,False Positive,CUST01325,0.956940,1,0,186,0,0.00,0.0,0.000,0,0.0,1,0,43,NaN,Yes
2,False Positive,CUST01411,0.936535,1,0,183,0,0.00,0.0,0.000,0,0.0,0,0,51,NaN,No
3,False Positive,CUST00437,0.934741,1,0,151,1,729.22,0.0,0.250,0,0.0,0,0,33,Silver,Yes
4,False Positive,CUST01370,0.890963,1,0,161,2,1246.04,0.0,0.215,0,0.0,2,0,35,NaN,No
5,False Positive,CUST01017,0.887623,1,0,133,2,1167.28,0.5,0.235,0,0.0,3,0,13,NaN,Yes
6,False Negative,CUST02072,0.047280,0,1,35,7,4340.19,0.0,0.420,0,0.0,4,0,1,NaN,Yes
7,False Negative,CUST00184,0.068322,0,1,14,3,2456.91,0.0,0.287,0,0.0,6,0,6,Platinum,No
8,False Negative,CUST01990,0.084019,0,1,59,4,3877.77,0.0,0.225,0,0.0,11,1,7,Silver,Yes
9,False Negative,CUST00866,0.117481,0,1,26,1,1280.71,0.0,0.060,0,0.0,5,0,1,NaN,No


## Saved Model Scoring Example

In [11]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -35, 35)))

def transform_with_bundle(df, bundle):
    parts = []
    for col in bundle['numeric_cols']:
        x = pd.to_numeric(df[col], errors='coerce').fillna(bundle['medians'][col]).to_numpy(float)
        parts.append(((x - bundle['means'][col]) / bundle['stds'][col]).reshape(-1, 1))
    for col in bundle['categorical_cols']:
        vals = df[col].fillna('Missing').astype(str)
        for cat in bundle['categories'][col]:
            parts.append((vals == cat).astype(float).to_numpy().reshape(-1, 1))
    X = np.hstack(parts)
    return np.hstack([np.ones((len(df), 1)), X])

with open('model.pkl', 'rb') as f:
    model = pickle.load(f)
example = test.head(5).copy()
proba = sigmoid(transform_with_bundle(example, model) @ np.array(model['weights']))
pd.DataFrame({'customer_id': example.customer_id, 'churn_probability': proba, 'predicted_churn': proba >= model['threshold']})

,customer_id,churn_probability,predicted_churn
15,CUST00016,0.965722,True
17,CUST00018,0.860944,True
23,CUST00024,0.125374,False
24,CUST00025,0.886813,True
29,CUST00030,0.049895,False


See `error_analysis.md` and `model_card.md` for business interpretation, limitations, ethics, and monitoring.